In [1]:
import sys

sys.path.append("..")

import warnings

import pandas as pd
import torch

import os

from distilrl.utils.misc import seed_everything
from distilrl.constants import CONFIG_DIR, DATASET_DIR, MODEL_DIR
from pathlib import Path

from datasets import Dataset

from hydra.utils import instantiate, get_class

warnings.filterwarnings("ignore")

%load_ext autoreload
%autoreload 2
%cd ..

device = "cuda" if torch.cuda.is_available() else "cpu"
device

/home/ivanov.dko/projects/distilrl


'cpu'

Тут не успеваю всё красиво оформить. Грусть, печаль. Но я думаю, всё в целом понятно. Собираем датасет, запускаем тренер, идём пить чай. Важно - тут нет поетри скрипта, поэтому придётся костыльно ставить все библиотеки ручками. В папку проекта можно убежать такими же командами, что в ранбуке

In [2]:
odd_set = Dataset.load_from_disk(Path(DATASET_DIR, "odd_set"), )
even_set = Dataset.load_from_disk(Path(DATASET_DIR, "even_set"))
uniform_set = Dataset.load_from_disk(Path(DATASET_DIR, "uniform_set"))

for dataset in [odd_set, even_set, uniform_set]:
    dataset.set_format(type="torch")

Отрезаем столько, сколько надо

In [ ]:
import numpy as np

seed_everything(42)
odd_set = odd_set.select(np.random.randint(0, odd_set.shape[0], 100000))
even_set = even_set.select(np.random.randint(0, even_set.shape[0], 5000))
uniform_set = uniform_set.select(np.random.randint(0, uniform_set.shape[0], 5000))

Функция, чтобы урезать длину датасета

In [ ]:
def trim_dataset(x, keys, seq_len=512):
    for key in keys:
        x[key] = x[key][:, -seq_len:]
    return x

In [3]:
from torch.utils.data import DataLoader

odd_loader = DataLoader(odd_set, batch_size=128, shuffle=True)
even_loader = DataLoader(even_set, batch_size=128, shuffle=False)
uniform_loader = DataLoader(uniform_set, batch_size=128, shuffle=False)

In [4]:
from distilrl.decision_tf.models import BanditDT
from distilrl.utils.data import load_pkl

distributions = load_pkl(Path(DATASET_DIR, "distributions.pkl"))
distributions = {k: torch.tensor(v).mean(axis=0) for k, v in distributions.items()}

dt = BanditDT(
    num_states=0,
    num_actions=10,
    seq_len=64,
    episode_len=20000,
    embedding_dim=128,
    num_layers=4,
    num_heads=4,
    attention_dropout=0.1,
    residual_dropout=0.1,
    embedding_dropout=0.1,
    feedforward_dim=512,
    max_action=1.0,
    emb_strategy="stack",
    optimizer=torch.optim.Adam,
    optimizer_kwargs=dict(lr=1e-4, betas=(0.9, 0.99)),
    scheduler=torch.optim.lr_scheduler.CosineAnnealingWarmRestarts,
    scheduler_kwargs=dict(T_0=1000, eta_min=2e-6),
    arm_distributions=distributions,
)

/home/ivanov.dko/projects/distilrl/.venv/lib/python3.10/site-packages/wandb/analytics/sentry.py:90: SentryHubDeprecationWarning: `sentry_sdk.Hub` is deprecated and will be removed in a future major release. Please consult our 1.x to 2.x migration guide for details on how to migrate `Hub` usage to the new API: https://docs.sentry.io/platforms/python/migration/1.x-to-2.x
  self.hub = sentry_sdk.Hub(client)


Этим можно воспользоваться уже, как чем-то вроде средней награды, а не пытаться просимулировать поведение бандита с такими то вероятностими, просто потому что  $E[Ber(p)] = p$

In [44]:
distributions["odd"]

tensor([0.2679, 0.7275, 0.2870, 0.7303, 0.2684, 0.7625, 0.2633, 0.7284, 0.3089,
        0.6961])

Можно потестить, что лоадер работает что с состояниями, что бещ

In [7]:
batch = next(iter(odd_loader))

In [ ]:
out = dt.forward(
    actions=batch["actions"],
    time_steps=batch["time_steps"],
    rewards=batch["rewards"],
    # states = states
)

Дальше только запустить с нужными параметрами

In [5]:
import pytorch_lightning as pl

trainer = pl.Trainer(
    enable_model_summary=True,
    accelerator="cpu",
    max_steps=10000,
    logger=None,
    gradient_clip_val=1.0,
    gradient_clip_algorithm="norm",
)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [ ]:
trainer.fit(dt, odd_loader, [even_loader, uniform_loader])